# PaperOrchestra — Multi-Agent Research Paper Writing Demo

This notebook demonstrates a multi-agent pipeline inspired by [PaperOrchestra](https://yiwen-song.github.io/paper_orchestra/) (Song et al., 2026) for automated AI research paper writing.

| Agent | Role |
|-------|------|
| **Outline Agent** | Structures raw materials into a paper plan with search queries |
| **Literature Review Agent** | Searches the web for related work, builds citation registry, drafts Intro + Related Work |
| **Section Writer Agent** | Drafts Abstract, Methodology, Experiments, Conclusion |
| **Refinement Agent** | Reviews the manuscript and iteratively improves it |

Orchestrated by **LangGraph** with configurable LLM backend (OpenAI / Anthropic / Google).

---

## 1. Install Dependencies

In [ ]:
%pip install -q -r requirements.txt

## 2. Configuration

Choose **one** method below.

### Method A — `.env` file

In [ ]:
from config.settings import PipelineConfig

# Reads from .env in the research_paper_creation/ directory
config = PipelineConfig.from_env()

### Method B — Hardcoded values

Run this cell **instead of** Method A.

In [ ]:
from config.settings import PipelineConfig

config = PipelineConfig.from_values(
    llm_provider="openai",             # "openai" | "anthropic" | "google" | "google_vertex"
    llm_model="gpt-4o",               # model name for your chosen provider
    search_provider="tavily",           # "tavily" for real search, "mock" for demo
    tavily_api_key="your-tavily-key",   # get one free at tavily.com
    max_refinement_rounds=2,
)

### Method C — Mock mode (no API keys needed)

Uses mock search results. Good for testing the pipeline structure.

In [ ]:
from config.settings import PipelineConfig

config = PipelineConfig.from_values(
    llm_provider="openai",             # still needs an LLM API key
    llm_model="gpt-4o",
    search_provider="mock",            # ← no Tavily key needed
    max_refinement_rounds=1,
)

---

## 3. Build the Workflow

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)-40s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)

from workflow.graph import build_paper_workflow

graph = build_paper_workflow(config)
print("Workflow compiled. Nodes:", list(graph.get_graph().nodes.keys()))

### Visualize the graph (optional)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"Graph visualization not available: {exc}")

---

## 4. Load Sample Inputs

The `examples/sample_inputs/` folder contains a sample idea summary and experimental log
for a fictional paper on adaptive sparse attention.

In [ ]:
from pathlib import Path

examples_dir = Path("examples/sample_inputs")

idea_summary = (examples_dir / "idea_summary.md").read_text()
experimental_log = (examples_dir / "experimental_log.md").read_text()
conference_guidelines = (examples_dir / "conference_guidelines.md").read_text()

print(f"Idea summary: {len(idea_summary)} chars")
print(f"Experimental log: {len(experimental_log)} chars")
print(f"Conference guidelines: {len(conference_guidelines)} chars")

---

## 5. Run the Pipeline

This invokes the full 4-agent pipeline:

1. **Outline Agent** — structures the paper plan
2. **Literature Review Agent** — searches for related work, drafts Intro + Related Work
3. **Section Writer Agent** — drafts Abstract, Methodology, Experiments, Conclusion
4. **Refinement Agent** — reviews and iteratively improves the manuscript

In [ ]:
result = graph.invoke({
    "idea_summary": idea_summary,
    "experimental_log": experimental_log,
    "conference_guidelines": conference_guidelines,
})

print(f"Status: {result['status']}")
print(f"Refinement rounds: {result.get('refinement_round', 0)}")
print(f"Manuscript length: {len(result['final_manuscript'])} characters")
print(f"Citations found: {len(result.get('citations', []))}")

---

## 6. View the Generated Manuscript

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result["final_manuscript"]))

## 7. Inspect Intermediate Outputs

In [ ]:
import json

print("=" * 60)
print("OUTLINE")
print("=" * 60)
print(json.dumps(result.get("outline", {}), indent=2)[:2000])

print("\n" + "=" * 60)
print(f"CITATIONS ({len(result.get('citations', []))} total)")
print("=" * 60)
for c in result.get("citations", [])[:5]:
    print(f"  [{c.get('year', '?')}] {c.get('authors', '?')} — {c.get('title', '?')}")

print("\n" + "=" * 60)
print("REVIEW FEEDBACK")
print("=" * 60)
feedback = result.get("review_feedback", "None")
print(feedback[:1000] if feedback else "No issues found — passed on first review.")

## 8. Save the Manuscript to Disk

In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "generated_paper.md"
output_path.write_text(result["final_manuscript"])
print(f"Manuscript saved to {output_path} ({len(result['final_manuscript'])} chars)")